# 01 - Plain text NER

Run GLiNER over `data/plain/validation.jsonl`, save predictions, and evaluate them with strict matching.

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# Para que leea la carpeta de antes pq pycharm me pide premium para modificar el archivo jupyter
import sys
sys.path.append('..')

from ner.evaluation import evaluate_ner_strict, evaluate_ner_relaxed
from ner.inference import load_gliner_model, predict_dataset
from ner.utils import load_jsonl, save_responses_to_jsonl

C:\Users\garci\Desktop\Projecte_Name_Entity_Recognition-master\Projecte_Name_Entity_Recognition-master\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [4]:
VALIDATION_PATH = '../data/plain/validation.jsonl'
OUTPUT_PATH = '../results/plain/gliner_small_validation.jsonl'

LABELS = [
    'PERSON_NAME', 'ID_CARD_NUMBER', 'ADDRESS',
    'REGISTRATION_NUMBER', 'DATE', 'AMOUNT', 'ORGANIZATION'
]


In [5]:
samples = load_jsonl(VALIDATION_PATH)
model = load_gliner_model(device='cpu') # cambiar cpu por cuda si no funciona por cpu
responses = predict_dataset(model, samples, LABELS, threshold=0.5)
save_responses_to_jsonl(responses, OUTPUT_PATH)


C:\Users\garci\Desktop\Projecte_Name_Entity_Recognition-master\Projecte_Name_Entity_Recognition-master\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

In [6]:
strict_results = evaluate_ner_strict(VALIDATION_PATH, OUTPUT_PATH)
strict_results['micro']



STRICT EVALUATION
Micro -> P: 0.73 | R: 0.84 | F1: 0.78 | TP: 1723 | FP: 649 | FN: 327

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 0.97   1.00   0.98     499      17       1
ID_CARD_NUMBER              0.75   0.94   0.84     188      62      12
ADDRESS                     0.00   0.00   0.00       0     417     200
REGISTRATION_NUMBER         0.93   0.76   0.84     190      14      60
DATE                        1.00   1.00   1.00     400       0       0
AMOUNT                      0.82   0.87   0.85     174      37      26
ORGANIZATION                0.73   0.91   0.81     272     102      28


{'precision': 0.7263912310286678,
 'recall': 0.8404878048780487,
 'f1': 0.7792853912256898,
 'true_positives': 1723,
 'false_positives': 649,
 'false_negatives': 327}

In [7]:
#TASK 1
strict_results = evaluate_ner_strict(VALIDATION_PATH, OUTPUT_PATH)
relaxed_results = evaluate_ner_relaxed(VALIDATION_PATH, OUTPUT_PATH)


STRICT EVALUATION
Micro -> P: 0.73 | R: 0.84 | F1: 0.78 | TP: 1723 | FP: 649 | FN: 327

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 0.97   1.00   0.98     499      17       1
ID_CARD_NUMBER              0.75   0.94   0.84     188      62      12
ADDRESS                     0.00   0.00   0.00       0     417     200
REGISTRATION_NUMBER         0.93   0.76   0.84     190      14      60
DATE                        1.00   1.00   1.00     400       0       0
AMOUNT                      0.82   0.87   0.85     174      37      26
ORGANIZATION                0.73   0.91   0.81     272     102      28

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.73 | R: 0.84 | F1: 0.78 | TP: 1723 | FP: 649 | FN: 327

ENTITY TYPE                    P      R     F1      TP      FP      FN
---------------------------------------------------------------------------

In [ ]:
# Diccionario donde guardaremos los resultados de cada modelo para compararlos luego
# Empezamos metiendo ya los del modelo small, que ejecutamos antes
all_results = {
    "gliner_small": {
        "strict": strict_results["micro"],                        # métricas estrictas (global)
        "relaxed_occ": relaxed_results["occurrences"]["micro"],   # relajada contando repeticiones
        "relaxed_uni": relaxed_results["unique"]["micro"],        # relajada sin repeticiones
    }
}

# Modelos nuevos que queremos probar
# La clave es el nombre corto (lo usamos para el archivo) y el valor es el nombre en Hugging Face
MODELS = {
    "gliner_medium": "gliner-community/gliner_medium-v2.5",
    "gliner_multi": "urchade/gliner_multi-v2.1",
}

# Repetimos el mismo proceso para cada modelo
for name, model_id in MODELS.items():
    print(f"\n##### {name} #####")   # título para saber qué modelo se está ejecutando

    # Cargamos el modelo (la primera vez lo descarga de internet)
    model = load_gliner_model(model_id)

    # Predecimos las entidades de todos los textos de validación
    responses = predict_dataset(model, samples, LABELS, threshold=0.5)

    # Cada modelo guarda sus predicciones en un archivo distinto, para no pisar los anteriores
    out_path = f"../results/plain/{name}_validation.jsonl"
    save_responses_to_jsonl(responses, out_path)

    # Evaluamos las predicciones con la evaluación estricta y la relajada
    strict = evaluate_ner_strict(VALIDATION_PATH, out_path)
    relaxed = evaluate_ner_relaxed(VALIDATION_PATH, out_path)

    # Guardamos solo los números globales (micro) de este modelo
    all_results[name] = {
        "strict": strict["micro"],
        "relaxed_occ": relaxed["occurrences"]["micro"],
        "relaxed_uni": relaxed["unique"]["micro"],
    }

    # Borramos el modelo de la memoria antes de pasar al siguiente
    del model


##### gliner_medium #####


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

C:\Users\garci\Desktop\Projecte_Name_Entity_Recognition-master\Projecte_Name_Entity_Recognition-master\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\garci\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\garci\Desktop\Projecte_Name_Entity_Recognition-master\Projec

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

C:\Users\garci\Desktop\Projecte_Name_Entity_Recognition-master\Projecte_Name_Entity_Recognition-master\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\garci\.cache\huggingface\hub\models--microsoft--deberta-v3-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


... (5 outputs left out)

C:\Users\garci\Desktop\Projecte_Name_Entity_Recognition-master\Projecte_Name_Entity_Recognition-master\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\garci\.cache\huggingface\hub\models--microsoft--mdeberta-v3-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spm.model: downloading bytes:           |  0.00B            

In [ ]:
# Cabecera de la tabla (F1 de cada tipo de evaluación)
print(f"{'MODELO':<15} {'STRICT F1':>10} {'REL-OCC F1':>11} {'REL-UNI F1':>11}")

# Una fila por modelo, con sus F1 redondeados a 2 decimales
for name, r in all_results.items():
    print(f"{name:<15} {r['strict']['f1']:>10.2f} {r['relaxed_occ']['f1']:>11.2f} {r['relaxed_uni']['f1']:>11.2f}")

In [ ]:
# Probar si la ruta del uv esta correcta porque en windows es un poco costron
import sys
print(sys.executable)

In [ ]:
import os
print(os.path.abspath(OUTPUT_PATH))
print(os.path.getsize(OUTPUT_PATH), "bytes")
print(len(samples), len(responses))

In [ ]:
import os
print(os.path.abspath(VALIDATION_PATH))
print(os.path.exists(VALIDATION_PATH))
print(os.path.getsize(VALIDATION_PATH), "bytes")